In [39]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, r2_score, confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mode

class DecisionTreeCART:
    def __init__(self, max_depth=100, min_samples=2, metric="gini", task="classification"):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.metric = metric
        self.task = task.lower()
        self.tree = None
        self._y_dtype = None
        self._num_all_samples = None

        self._supported_metrics = {
            "classification": {
                "gini": self._calculate_gini,
                "entropy": self._calculate_entropy,
            },
            "regression": {
                "mse": self._calculate_mse,
                "mae": self._calculate_mae,
            },
        }

    def _set_df_type(self, X, y, dtype):
        X = X.astype(dtype)
        y = y.astype(dtype) if self.task == "regression" else y
        self._y_dtype = y.dtype
        return X, y

    @staticmethod
    def _purity(y):
        return len(np.unique(y)) == 1

    @staticmethod
    def _is_leaf_node(node):
        return not isinstance(node, dict)

    def _leaf_node(self, y):
        if self.task == "regression":
            return float(np.mean(y))
        
        try:
            mod = mode(y)
            if isinstance(mod, tuple) or isinstance(mod, np.ndarray):
                return mod[0][0] if len(mod[0]) > 0 else y.iloc[0]
            return mod
        except:
            values, counts = np.unique(y, return_counts=True)
            return values[np.argmax(counts)]

    def _split_df(self, X, y, feature, threshold):
        left_mask = X[feature] <= threshold
        right_mask = ~left_mask
        left_idx, right_idx = X[left_mask].index, X[right_mask].index
        if len(left_idx) == 0 or len(right_idx) == 0:
            return self._leaf_node(y)
        return left_idx, right_idx

    def _get_metric_function(self):
        if callable(self.metric):
            return self.metric

        available_metrics = self._supported_metrics[self.task]
        if self.metric not in available_metrics:
            raise ValueError(
                f"Metric '{self.metric}' is not supported for task '{self.task}'. "
                f"Available: {list(available_metrics.keys())}"
            )
        return available_metrics[self.metric]

    @staticmethod
    def _calculate_gini(y):
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return 1 - np.sum(probs ** 2)

    @staticmethod
    def _calculate_entropy(y):
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return -np.sum(probs * np.log2(probs + 1e-10))

    @staticmethod
    def _calculate_mse(y):
        return np.mean((y - np.mean(y)) ** 2)

    @staticmethod
    def _calculate_mae(y):
        return np.mean(np.abs(y - np.mean(y)))

    def _cost_function(self, left_y, right_y):
        metric_func = self._get_metric_function()
        n_total = len(left_y) + len(right_y)
        p_left = len(left_y) / n_total
        p_right = len(right_y) / n_total
        return p_left * metric_func(left_y) + p_right * metric_func(right_y)

    def _node_error_rate(self, y):
        if self._num_all_samples is None:
            self._num_all_samples = len(y)
        metric_func = self._get_metric_function()
        return len(y) / self._num_all_samples * metric_func(y)

    def _best_split(self, X, y):
        best_feature, best_threshold = None, None
        min_cost = float('inf')

        for feature in X.columns:
            unique_values = np.unique(X[feature])
            for i in range(1, len(unique_values)):
                threshold = (unique_values[i-1] + unique_values[i]) / 2
                split_result = self._split_df(X, y, feature, threshold)
                if isinstance(split_result, tuple):
                    left_idx, right_idx = split_result
                    current_cost = self._cost_function(y[left_idx], y[right_idx])
                    if current_cost < min_cost:
                        min_cost = current_cost
                        best_feature = feature
                        best_threshold = threshold

        return best_feature, best_threshold

    def _stopping_conditions(self, y, depth, n_samples):
        return (
            self._purity(y),
            depth >= self.max_depth,
            n_samples < self.min_samples,
        )

    def _grow_tree(self, X, y, depth=0):
        n_samples = len(y)
        X, y = self._set_df_type(X, y, np.float64)

        if any(self._stopping_conditions(y, depth, n_samples)):
            error_rate = self._node_error_rate(y)
            return f"{self._leaf_node(y)} | error {error_rate:.3f}"

        best_feature, best_threshold = self._best_split(X, y)
        decision_node = f"{best_feature} <= {best_threshold} | as_leaf {self._leaf_node(y)}"

        left_idx, right_idx = self._split_df(X, y, best_feature, best_threshold)
        left_subtree = self._grow_tree(X.loc[left_idx], y.loc[left_idx], depth + 1)
        right_subtree = self._grow_tree(X.loc[right_idx], y.loc[right_idx], depth + 1)

        if left_subtree == right_subtree:
            return left_subtree
        return {decision_node: [left_subtree, right_subtree]}

    def fit(self, X: pd.DataFrame, y: pd.Series):
        X = X.reset_index(drop=True)
        y = y.reset_index(drop=True)
        self.tree = self._grow_tree(X, y)

    def _traverse_tree(self, sample, tree):
        if self._is_leaf_node(tree):
            return tree.split(" | ")[0]

        decision_node = next(iter(tree))
        feature, threshold = decision_node.split(" <= ")[0], float(decision_node.split(" <= ")[1].split(" |")[0])
        left_subtree, right_subtree = tree[decision_node]

        if sample[feature] <= threshold:
            return self._traverse_tree(sample, left_subtree)
        return self._traverse_tree(sample, right_subtree)

    def predict(self, X: pd.DataFrame):
        return np.array([self._traverse_tree(row, self.tree) for _, row in X.iterrows()], dtype=self._y_dtype)

In [41]:
data_reg = pd.read_csv('/Users/macbook/Desktop/ВУЗ/Машинное обучение и большие данные/lab_1/mumbai_houses.csv')

In [42]:
X = data_reg.drop(columns=['price'])
y = data_reg['price']

In [43]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

In [44]:
tree = DecisionTreeCART(task='regression', metric='mse')
tree.fit(X_train, y_train)

In [45]:
predictions = tree.predict(X_test)

In [46]:
r2_score(y_test, predictions)

0.9962996136519809

In [47]:
data_class = pd.read_csv('/Users/macbook/Desktop/ВУЗ/Машинное обучение и большие данные/lab_1/heart.csv')

In [48]:
X = data_class.drop(columns=['num'])
y = data_class['num']

In [49]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [50]:
tree = DecisionTreeCART(task='classification', metric='gini')
tree.fit(X_train, y_train)

In [51]:
predictions = tree.predict(X_test)

In [52]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.84      0.75      0.79        36
           1       0.12      0.11      0.12         9
           2       0.29      0.57      0.38         7
           3       0.00      0.00      0.00         5
           4       0.00      0.00      0.00         2

    accuracy                           0.54        59
   macro avg       0.25      0.29      0.26        59
weighted avg       0.57      0.54      0.55        59



/Users/macbook/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/macbook/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/macbook/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

In [53]:
print(confusion_matrix(y_test, predictions))

[[27  3  5  1  0]
 [ 3  1  3  2  0]
 [ 0  2  4  1  0]
 [ 1  2  2  0  0]
 [ 1  0  0  1  0]]
